# UniAD 2.0 Evaluation on Google Colab

This notebook runs UniAD Stage 1 (Track & Map) evaluation on Colab's free T4 GPU.

**⚠️ Compatibility Note:**  
UniAD requires Python 3.9 + mmcv-full 1.6.1, but Colab now uses Python 3.12.  
This notebook uses a workaround to install compatible packages.

**Requirements:**
- Google account with Google Drive
- nuScenes dataset (or mini version for testing)

**Estimated Time:** 
- First run: ~20-30 min setup + 2-4 hours eval
- Subsequent runs: ~2-5 min setup (cached) + 2-4 hours eval

## 1. Check GPU

In [ ]:
# Verify GPU is available
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create cache directories on Google Drive for persistence
!mkdir -p /content/drive/MyDrive/colab_cache/pip
!mkdir -p /content/drive/MyDrive/colab_cache/UniAD_ckpts

## 3. Clone UniAD Repository

In [ ]:
%cd /content
!git clone https://github.com/RobotMa/UniAD.git
%cd UniAD

## 4. Install Dependencies

**First run workflow:**
1. Run install cells → See "Restart session" warning
2. Click "Restart session" 
3. After restart, **skip to Section 5** (packages are installed)

**Subsequent runs:** Skip install cells entirely if cache restored

In [ ]:
import os

# Check if we have a cached environment
CACHE_DIR = "/content/drive/MyDrive/colab_cache"
PIP_CACHE = f"{CACHE_DIR}/pip"
SITE_PACKAGES_CACHE = f"{CACHE_DIR}/site_packages_py312.tar.gz"  # Version-specific cache

# Set pip cache directory to Google Drive
os.environ["PIP_CACHE_DIR"] = PIP_CACHE

# Check if cached site-packages exists
if os.path.exists(SITE_PACKAGES_CACHE):
    print("Found cached packages! Restoring...")
    !tar -xzf {SITE_PACKAGES_CACHE} -C /usr/local/lib/python3.12/dist-packages/ 2>/dev/null || true
    print("Cache restored! Verifying installation...")
    !python -c "import torch; import mmcv; import mmdet; print('All packages loaded successfully!')"
else:
    print("No cache found. Installing packages (this will be cached for next time)...")
    print("Run the cells below to install dependencies.")

In [ ]:
# ============================================================
# FIRST RUN ONLY - Skip this cell if cache was restored above
# ============================================================

# Check current environment
import sys
print(f"Python version: {sys.version}")
!nvcc --version | grep release

# Install PyTorch 2.2 (oldest version available for Python 3.12 + CUDA 11.8)
!pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu118

# Build mmcv-full from source (no pre-built wheels for Python 3.12)
print("\nBuilding mmcv-full from source (this takes ~10 min)...")
!pip install -U pip setuptools wheel
!pip install addict yapf numpy==1.26.4 Pillow pyyaml

%cd /content
!git clone https://github.com/open-mmlab/mmcv.git -b v1.7.2
%cd mmcv
!MMCV_WITH_OPS=1 pip install -e . -v

In [ ]:
# Install mmdet and mmseg from source (compatible versions)
%cd /content

# mmdetection 2.x
!git clone https://github.com/open-mmlab/mmdetection.git -b v2.28.2
%cd mmdetection
!pip install -e . -v

# mmsegmentation
%cd /content
!git clone https://github.com/open-mmlab/mmsegmentation.git -b v0.30.0
%cd mmsegmentation
!pip install -e . -v

# nuscenes-devkit
!pip install nuscenes-devkit motmetrics

In [ ]:
# Install mmdet3d from source
%cd /content
!git clone https://github.com/open-mmlab/mmdetection3d.git -b v1.0.0rc6
%cd mmdetection3d
!pip install -e . -v

# Install UniAD
%cd /content/UniAD
!pip install -r requirements.txt
!pip install -e .

print("\n" + "="*60)
print("Installation complete!")
print("Click 'Restart session' if prompted, then skip to Section 5.")
print("="*60)

In [ ]:
# Save installed packages to Google Drive cache (run once after first install)
# This creates a ~5-8GB cache file but saves 15+ min on future runs

import os
SITE_PACKAGES_CACHE = "/content/drive/MyDrive/colab_cache/site_packages_py312.tar.gz"

if not os.path.exists(SITE_PACKAGES_CACHE):
    print("Saving packages to Google Drive cache (one-time, ~5 min)...")
    !tar -czf {SITE_PACKAGES_CACHE} -C /usr/local/lib/python3.12/dist-packages/ . 2>/dev/null
    print(f"Cache saved! Size: ")
    !ls -lh {SITE_PACKAGES_CACHE}
else:
    print("Cache already exists, skipping save.")

## 5. Download Pretrained Checkpoints

In [ ]:
%cd /content/UniAD
!mkdir -p ckpts

CKPT_CACHE = "/content/drive/MyDrive/colab_cache/UniAD_ckpts"

# Check if checkpoints already cached on Drive
import os
if os.path.exists(f"{CKPT_CACHE}/uniad_base_track_map.pth"):
    print("Found cached checkpoints on Drive! Linking...")
    !ln -sf {CKPT_CACHE}/uniad_base_track_map.pth ckpts/
    !ln -sf {CKPT_CACHE}/bevformer_r101_dcn_24ep.pth ckpts/
else:
    print("Downloading checkpoints (will be cached to Drive)...")
    %cd {CKPT_CACHE}
    !wget -q --show-progress https://huggingface.co/OpenDriveLab/UniAD2.0_R101_nuScenes/resolve/main/ckpts/uniad_base_track_map.pth
    !wget -q --show-progress https://huggingface.co/OpenDriveLab/UniAD2.0_R101_nuScenes/resolve/main/ckpts/bevformer_r101_dcn_24ep.pth
    %cd /content/UniAD
    !ln -sf {CKPT_CACHE}/uniad_base_track_map.pth ckpts/
    !ln -sf {CKPT_CACHE}/bevformer_r101_dcn_24ep.pth ckpts/

print("\nCheckpoints ready:")
!ls -lh ckpts/

## 6. Setup nuScenes Dataset

**Option A:** Link from Google Drive (if you uploaded nuScenes there)

**Option B:** Use nuScenes mini for quick testing

In [ ]:
# Option A: Link nuScenes from Google Drive
# Uncomment and modify the path if you have nuScenes on Drive

# !mkdir -p /content/UniAD/data
# !ln -s /content/drive/MyDrive/nuscenes /content/UniAD/data/nuscenes

In [ ]:
# Option B: Download nuScenes mini (for testing only - ~4GB)
# Note: Mini dataset is for testing the pipeline, not full evaluation

# Uncomment to download mini dataset:
# !mkdir -p /content/UniAD/data/nuscenes
# %cd /content/UniAD/data/nuscenes
# !wget https://www.nuscenes.org/data/v1.0-mini.tgz
# !tar -xzf v1.0-mini.tgz

## 7. Prepare Data PKL Files

If you don't have the preprocessed PKL files, run data preparation:

In [ ]:
# Generate data info files (only needed once)
# Skip this if you already have the .pkl files

%cd /content/UniAD

# Uncomment to run data preparation:
# !python tools/create_data.py nuscenes --root-path ./data/nuscenes --out-dir ./data/nuscenes --extra-tag nuscenes

## 8. Run Evaluation

In [ ]:
%cd /content/UniAD

# Run Stage 1 evaluation with 1 GPU
!python tools/test.py \
    projects/configs/stage1_track_map/base_track_map.py \
    ckpts/uniad_base_track_map.pth \
    --eval bbox

## Expected Results

If everything works correctly, you should see:
```
Aggregated results: 
AMOTA    0.394 
AMOTP    1.316
RECALL   0.484
```

## Troubleshooting

**Out of Memory:**
- Try reducing `queue_length` from 5 to 3 in the config
- Use Colab Pro for A100 GPU

**Missing files:**
- Ensure nuScenes dataset is properly linked
- Check that all .pkl files exist in data/nuscenes/

**Dependency errors:**
- Restart runtime and run cells from beginning